# SBV2 01 - Data Download And Manifest

Purpose: stage the English-first Sandbox V2 document sources and write a reviewable acquisition manifest before NYPL sampling begins.

Minor correction carried forward from Notebook 0 review: this notebook prioritises SROIE and FUNSD first. CORD is optional and off by default. Korean menus remain future/OOD work and are not part of the first English benchmark.

## Notebook Contract

Expected visible outputs:

- source policy table
- download/load configuration
- SROIE and FUNSD acquisition counts
- sample manifest rows with local image paths
- NYPL handoff status for `SBV2_02_nypl_sampling.ipynb`
- audit CSV/JSON files under `outputs/sandbox_v2/audit_tables/`

This notebook does not run OCR, VLM extraction, generation, or heavyweight model loading.

In [1]:
from __future__ import annotations

import json
import os
import re
import shutil
import sys
from datetime import datetime
from pathlib import Path
from typing import Any

import pandas as pd
from IPython.display import display
from PIL import Image

try:
    from datasets import load_dataset
except Exception as exc:
    load_dataset = None
    DATASETS_IMPORT_ERROR = repr(exc)
else:
    DATASETS_IMPORT_ERROR = ""

ROOT = Path.cwd().resolve()
if not (ROOT / "sandbox_v2").exists():
    candidates = [p for p in ROOT.parents if (p / "sandbox_v2").exists()]
    if candidates:
        ROOT = candidates[0]

os.chdir(ROOT)

CACHE_DIR = Path(os.environ.get("MENUFORGE_CACHE_HOME", ROOT / ".cache")).resolve()
os.environ["XDG_CACHE_HOME"] = str(CACHE_DIR)
os.environ["PADDLE_PDX_CACHE_HOME"] = str(CACHE_DIR / "paddlex")
os.environ["MPLCONFIGDIR"] = str(CACHE_DIR / "matplotlib")
os.environ["PIP_CACHE_DIR"] = str(CACHE_DIR / "pip")
os.environ["HF_HOME"] = str(CACHE_DIR / "huggingface")
os.environ["HF_HUB_CACHE"] = str(Path(os.environ["HF_HOME"]) / "hub")
os.environ["HF_XET_CACHE"] = str(Path(os.environ["HF_HOME"]) / "xet")
os.environ["TORCH_HOME"] = str(CACHE_DIR / "torch")
os.environ["IPYTHONDIR"] = str(CACHE_DIR / "ipython")
os.environ["JUPYTER_CONFIG_DIR"] = str(CACHE_DIR / "jupyter" / "config")
os.environ["JUPYTER_DATA_DIR"] = str(CACHE_DIR / "jupyter" / "data")
os.environ["JUPYTER_RUNTIME_DIR"] = str(CACHE_DIR / "jupyter" / "runtime")

for key in [
    "PADDLE_PDX_CACHE_HOME",
    "MPLCONFIGDIR",
    "PIP_CACHE_DIR",
    "HF_HOME",
    "HF_HUB_CACHE",
    "HF_XET_CACHE",
    "TORCH_HOME",
    "IPYTHONDIR",
    "JUPYTER_CONFIG_DIR",
    "JUPYTER_DATA_DIR",
    "JUPYTER_RUNTIME_DIR",
]:
    Path(os.environ[key]).mkdir(parents=True, exist_ok=True)

RAW_DIR = ROOT / "data" / "raw" / "sandbox_v2"
INTERIM_MANIFEST_DIR = ROOT / "data" / "interim" / "sandbox_v2" / "manifests"
AUDIT_DIR = ROOT / "outputs" / "sandbox_v2" / "audit_tables"
for directory in [RAW_DIR, INTERIM_MANIFEST_DIR, AUDIT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

RUN_TIMESTAMP = datetime.now().isoformat(timespec="seconds")

print(f"Project root: {ROOT}")
print(f"Raw Sandbox V2 dir: {RAW_DIR}")
print(f"Manifest dir: {INTERIM_MANIFEST_DIR}")
print(f"Audit dir: {AUDIT_DIR}")
print(f"HF cache: {os.environ['HF_HOME']}")

Project root: /home/endy/menuforge
Raw Sandbox V2 dir: /home/endy/menuforge/data/raw/sandbox_v2
Manifest dir: /home/endy/menuforge/data/interim/sandbox_v2/manifests
Audit dir: /home/endy/menuforge/outputs/sandbox_v2/audit_tables
HF cache: /home/endy/menuforge/.cache/huggingface


## Source Policy

The first successful pass stays English-first and preprocessing-first.

In [2]:
source_policy_df = pd.DataFrame(
    [
        {
            "source_id": "sroie",
            "register_id": "data-sroie",
            "role": "core English OCR/key-field source",
            "notebook_1_action": "download/load first",
            "claim_boundary": "English receipt OCR/layout evidence",
        },
        {
            "source_id": "funsd",
            "register_id": "data-funsd",
            "role": "core English form/layout source",
            "notebook_1_action": "download/load first",
            "claim_boundary": "English noisy-document layout/entity evidence",
        },
        {
            "source_id": "nypl",
            "register_id": "data-nypl",
            "role": "actual menu source",
            "notebook_1_action": "verify local handoff only",
            "claim_boundary": "sample and inspect in SBV2_02",
        },
        {
            "source_id": "cord",
            "register_id": "data-cord",
            "role": "optional structure stress test",
            "notebook_1_action": "deferred unless SBV2_INCLUDE_CORD=1",
            "claim_boundary": "do not use for English generation claims",
        },
        {
            "source_id": "korean_menus",
            "register_id": "data-korean-menu",
            "role": "future multilingual/OOD sidecar",
            "notebook_1_action": "deferred",
            "claim_boundary": "not part of first English benchmark",
        },
    ]
)

display(source_policy_df)

,source_id,register_id,role,notebook_1_action,claim_boundary
0,sroie,data-sroie,core English OCR/key-field source,download/load first,English receipt OCR/layout evidence
1,funsd,data-funsd,core English form/layout source,download/load first,English noisy-document layout/entity evidence
2,nypl,data-nypl,actual menu source,verify local handoff only,sample and inspect in SBV2_02
3,cord,data-cord,optional structure stress test,deferred unless SBV2_INCLUDE_CORD=1,do not use for English generation claims
4,korean_menus,data-korean-menu,future multilingual/OOD sidecar,deferred,not part of first English benchmark


## Acquisition Configuration

Defaults are intentionally modest. Override with environment variables before launching Jupyter if you want a larger pass:

- `SBV2_SROIE_LIMIT=300`
- `SBV2_FUNSD_LIMIT=199`
- `SBV2_INCLUDE_CORD=1`
- `SBV2_CORD_LIMIT=50`
- `SBV2_OFFLINE_SMOKE_ONLY=1`

In [3]:
def env_int(name: str, default: int) -> int:
    raw = os.environ.get(name, "").strip()
    if not raw:
        return default
    try:
        return int(raw)
    except ValueError:
        print(f"Warning: {name}={raw!r} is not an integer; using {default}.")
        return default


CONFIG = {
    "sroie_limit": env_int("SBV2_SROIE_LIMIT", 300),
    "funsd_limit": env_int("SBV2_FUNSD_LIMIT", 199),
    "include_cord": os.environ.get("SBV2_INCLUDE_CORD", "0").strip() == "1",
    "cord_limit": env_int("SBV2_CORD_LIMIT", 50),
    "offline_smoke_only": os.environ.get("SBV2_OFFLINE_SMOKE_ONLY", "0").strip() == "1",
}

config_df = pd.DataFrame([{"setting": key, "value": value} for key, value in CONFIG.items()])
display(config_df)

,setting,value
0,sroie_limit,300
1,funsd_limit,199
2,include_cord,False
3,cord_limit,50
4,offline_smoke_only,False


## Helpers

These helpers keep the notebook tolerant of small schema differences across Hugging Face dataset rows.

In [4]:
def safe_slug(value: Any, fallback: str) -> str:
    text = str(value or fallback).strip().lower()
    text = re.sub(r"[^a-z0-9_.-]+", "-", text)
    text = text.strip("-._")
    return text or fallback


def jsonable(value: Any) -> Any:
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {str(k): jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [jsonable(v) for v in value]
    if hasattr(value, "tolist"):
        try:
            return value.tolist()
        except Exception:
            pass
    if isinstance(value, Image.Image):
        return "<PIL.Image>"
    return str(value)


def pick_image(row: dict[str, Any]) -> tuple[Image.Image | None, str]:
    for key in ["image", "img", "document", "page_image"]:
        value = row.get(key)
        if isinstance(value, Image.Image):
            return value, key
    for key, value in row.items():
        if isinstance(value, Image.Image):
            return value, key
    for key in ["image_path", "path", "file_name", "filename"]:
        value = row.get(key)
        if value:
            candidate = Path(value)
            if candidate.exists():
                return Image.open(candidate), key
    return None, ""


def extract_words(row: dict[str, Any]) -> list[str]:
    for key in ["words", "tokens", "text", "texts", "ocr_tokens"]:
        value = row.get(key)
        if value is None:
            continue
        if isinstance(value, str):
            return value.split()
        if isinstance(value, (list, tuple)):
            return [str(item) for item in value]
    return []


def extract_bboxes(row: dict[str, Any]) -> list[Any]:
    for key in ["bboxes", "boxes", "bbox", "bounding_boxes", "word_bboxes"]:
        value = row.get(key)
        if isinstance(value, (list, tuple)):
            return list(value)
    return []


def extract_entity_keys(row: dict[str, Any]) -> list[str]:
    for key in ["entities", "entity", "labels", "ner_tags", "tags"]:
        value = row.get(key)
        if isinstance(value, dict):
            return sorted(str(k) for k in value.keys())
        if isinstance(value, (list, tuple)):
            return sorted({str(item) for item in value[:50]})[:20]
    return []


def text_ratios(words: list[str]) -> tuple[int, float, float]:
    text = " ".join(words)
    if not text:
        return 0, 0.0, 0.0
    ascii_ratio = sum(ord(ch) < 128 for ch in text) / len(text)
    latin_letters = sum(("a" <= ch.lower() <= "z") for ch in text)
    latin_letter_ratio = latin_letters / len(text)
    return len(text), round(ascii_ratio, 3), round(latin_letter_ratio, 3)


def load_hf_split(dataset_name: str, split: str):
    if load_dataset is None:
        raise RuntimeError(f"datasets import failed: {DATASETS_IMPORT_ERROR}")
    try:
        return load_dataset(dataset_name, split=split, trust_remote_code=True)
    except (TypeError, ValueError):
        return load_dataset(dataset_name, split=split)


def smoke_fallback_rows(source_id: str, spec: dict[str, Any], reason: str) -> list[dict[str, Any]]:
    smoke_path = RAW_DIR / "smoke_sources" / "smoke_manifest.jsonl"
    if not smoke_path.exists():
        return []
    rows = []
    for line in smoke_path.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        item = json.loads(line)
        if item.get("source") != source_id:
            continue
        image_path = ROOT / item["image_path"]
        rows.append(
            {
                "source_id": source_id,
                "register_id": spec["register_id"],
                "dataset_name": spec["dataset_name"],
                "split": "smoke",
                "source_index": item.get("idx"),
                "document_id": f"{source_id}_smoke_{int(item.get('idx', 0)):05d}",
                "image_path": str(image_path.relative_to(ROOT)),
                "document_type": spec["document_type"],
                "licence": spec["licence"],
                "ground_truth_available": spec["ground_truth_available"],
                "annotation_fields": "smoke_manifest",
                "width": item.get("width"),
                "height": item.get("height"),
                "text_chars": item.get("text_chars"),
                "ascii_ratio": item.get("ascii_ratio"),
                "latin_letter_ratio": item.get("latin_letter_ratio"),
                "words_count": item.get("words_count", ""),
                "bboxes_count": item.get("bboxes_count", ""),
                "entity_keys": json.dumps(item.get("entity_keys", [])),
                "acquisition_status": f"smoke_fallback: {reason}",
                "accessed_at": RUN_TIMESTAMP,
            }
        )
    return rows


print("Helper functions ready.")

Helper functions ready.


## Core English Sources

SROIE and FUNSD are the only external datasets downloaded or staged by default in this notebook.

In [5]:
core_specs = [
    {
        "source_id": "sroie",
        "register_id": "data-sroie",
        "dataset_name": "jsdnrs/ICDAR2019-SROIE",
        "split": "train",
        "limit": CONFIG["sroie_limit"],
        "document_type": "receipt",
        "licence": "CC BY 4.0 per HF card; verify in final reporting",
        "ground_truth_available": True,
    },
    {
        "source_id": "funsd",
        "register_id": "data-funsd",
        "dataset_name": "nielsr/funsd",
        "split": "train",
        "limit": CONFIG["funsd_limit"],
        "document_type": "form",
        "licence": "verify exact dataset licence before reported results",
        "ground_truth_available": True,
    },
]


def acquire_hf_sample(spec: dict[str, Any]) -> tuple[list[dict[str, Any]], dict[str, Any]]:
    source_id = spec["source_id"]
    source_dir = RAW_DIR / source_id
    image_dir = source_dir / "images"
    annotation_dir = source_dir / "annotations"
    image_dir.mkdir(parents=True, exist_ok=True)
    annotation_dir.mkdir(parents=True, exist_ok=True)

    if CONFIG["offline_smoke_only"]:
        rows = smoke_fallback_rows(source_id, spec, "SBV2_OFFLINE_SMOKE_ONLY=1")
        return rows, {
            "source_id": source_id,
            "dataset_name": spec["dataset_name"],
            "split": "smoke",
            "requested_limit": spec["limit"],
            "rows_staged": len(rows),
            "status": "smoke_only",
            "message": "Offline smoke mode enabled.",
            "local_dir": str(source_dir.relative_to(ROOT)),
        }

    try:
        dataset = load_hf_split(spec["dataset_name"], spec["split"])
    except Exception as exc:
        reason = f"HF load failed: {type(exc).__name__}: {exc}"
        rows = smoke_fallback_rows(source_id, spec, reason)
        return rows, {
            "source_id": source_id,
            "dataset_name": spec["dataset_name"],
            "split": spec["split"],
            "requested_limit": spec["limit"],
            "rows_staged": len(rows),
            "status": "load_failed_used_smoke_fallback" if rows else "load_failed_no_fallback",
            "message": reason,
            "local_dir": str(source_dir.relative_to(ROOT)),
        }

    rows = []
    dataset_len = len(dataset) if hasattr(dataset, "__len__") else "unknown"
    limit = max(0, int(spec["limit"]))

    for source_index, row in enumerate(dataset):
        if source_index >= limit:
            break
        row = dict(row)
        image, image_key = pick_image(row)
        if image is None:
            continue
        image = image.convert("RGB")
        row_id = row.get("id") or row.get("file_name") or row.get("filename") or source_index
        document_id = f"{source_id}_{safe_slug(row_id, str(source_index).zfill(5))}"
        image_path = image_dir / f"{document_id}.jpg"
        annotation_path = annotation_dir / f"{document_id}.json"
        image.save(image_path, quality=92)

        words = extract_words(row)
        bboxes = extract_bboxes(row)
        entity_keys = extract_entity_keys(row)
        text_chars, ascii_ratio, latin_letter_ratio = text_ratios(words)
        annotation_payload = {key: value for key, value in row.items() if key != image_key}
        annotation_payload["_sbv2_image_key"] = image_key
        annotation_payload["_sbv2_source_index"] = source_index
        annotation_path.write_text(json.dumps(jsonable(annotation_payload), indent=2), encoding="utf-8")

        rows.append(
            {
                "source_id": source_id,
                "register_id": spec["register_id"],
                "dataset_name": spec["dataset_name"],
                "split": spec["split"],
                "source_index": source_index,
                "document_id": document_id,
                "image_path": str(image_path.relative_to(ROOT)),
                "annotation_path": str(annotation_path.relative_to(ROOT)),
                "document_type": spec["document_type"],
                "licence": spec["licence"],
                "ground_truth_available": spec["ground_truth_available"],
                "annotation_fields": json.dumps(sorted(key for key in row.keys() if key != image_key)),
                "width": image.width,
                "height": image.height,
                "text_chars": text_chars,
                "ascii_ratio": ascii_ratio,
                "latin_letter_ratio": latin_letter_ratio,
                "words_count": len(words),
                "bboxes_count": len(bboxes),
                "entity_keys": json.dumps(entity_keys),
                "acquisition_status": "downloaded_or_loaded_from_hf_cache",
                "accessed_at": RUN_TIMESTAMP,
            }
        )

    status = "ok" if rows else "no_rows_staged"
    return rows, {
        "source_id": source_id,
        "dataset_name": spec["dataset_name"],
        "split": spec["split"],
        "dataset_len": dataset_len,
        "requested_limit": limit,
        "rows_staged": len(rows),
        "status": status,
        "message": "",
        "local_dir": str(source_dir.relative_to(ROOT)),
    }


manifest_rows = []
acquisition_rows = []
for spec in core_specs:
    rows, status = acquire_hf_sample(spec)
    manifest_rows.extend(rows)
    acquisition_rows.append(status)

acquisition_df = pd.DataFrame(acquisition_rows)
manifest_df = pd.DataFrame(manifest_rows)

display(acquisition_df)
display(manifest_df.head(12))

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'jsdnrs/ICDAR2019-SROIE' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/319M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/191M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/626 [00:00<?, ? examples/s]

...

data/test-00000-of-00001.parquet:   0%|          | 0.00/4.38M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/149 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/50 [00:00<?, ? examples/s]

,source_id,dataset_name,split,dataset_len,requested_limit,rows_staged,status,message,local_dir
0,sroie,jsdnrs/ICDAR2019-SROIE,train,626,300,300,ok,,data/raw/sandbox_v2/sroie
1,funsd,nielsr/funsd,train,149,199,149,ok,,data/raw/sandbox_v2/funsd


,source_id,register_id,dataset_name,split,source_index,document_id,image_path,annotation_path,document_type,licence,...,width,height,text_chars,ascii_ratio,latin_letter_ratio,words_count,bboxes_count,entity_keys,acquisition_status,accessed_at
0,sroie,data-sroie,jsdnrs/ICDAR2019-SROIE,train,0,sroie_00000,data/raw/sandbox_v2/sroie/images/sroie_00000.jpg,data/raw/sandbox_v2/sroie/annotations/sroie_00...,receipt,CC BY 4.0 per HF card; verify in final reporting,...,463,1013,459,1.0,0.584,44,44,"[""address"", ""company"", ""date"", ""total""]",downloaded_or_loaded_from_hf_cache,2026-04-29T09:47:44
1,sroie,data-sroie,jsdnrs/ICDAR2019-SROIE,train,1,sroie_1,data/raw/sandbox_v2/sroie/images/sroie_1.jpg,data/raw/sandbox_v2/sroie/annotations/sroie_1....,receipt,CC BY 4.0 per HF card; verify in final reporting,...,439,1004,660,1.0,0.495,48,48,"[""address"", ""company"", ""date"", ""total""]",downloaded_or_loaded_from_hf_cache,2026-04-29T09:47:44
2,sroie,data-sroie,jsdnrs/ICDAR2019-SROIE,train,2,sroie_2,data/raw/sandbox_v2/sroie/images/sroie_2.jpg,data/raw/sandbox_v2/sroie/annotations/sroie_2....,receipt,CC BY 4.0 per HF card; verify in final reporting,...,459,949,698,1.0,0.496,54,54,"[""address"", ""company"", ""date"", ""total""]",downloaded_or_loaded_from_hf_cache,2026-04-29T09:47:44
3,sroie,data-sroie,jsdnrs/ICDAR2019-SROIE,train,3,sroie_3,data/raw/sandbox_v2/sroie/images/sroie_3.jpg,data/raw/sandbox_v2/sroie/annotations/sroie_3....,receipt,CC BY 4.0 per HF card; verify in final reporting,...,461,933,584,1.0,0.541,60,60,"[""address"", ""company"", ""date"", ""total""]",downloaded_or_loaded_from_hf_cache,2026-04-29T09:47:44
4,sroie,data-sroie,jsdnrs/ICDAR2019-SROIE,train,4,sroie_4,data/raw/sandbox_v2/sroie/images/sroie_4.jpg,data/raw/sandbox_v2/sroie/annotations/sroie_4....,receipt,CC BY 4.0 per HF card; verify in final reporting,...,463,1026,772,1.0,0.468,61,61,"[""address"", ""company"", ""date"", ""total""]",downloaded_or_loaded_from_hf_cache,2026-04-29T09:47:44
5,sroie,data-sroie,jsdnrs/ICDAR2019-SROIE,train,5,sroie_5,data/raw/sandbox_v2/sroie/images/sroie_5.jpg,data/raw/sandbox_v2/sroie/annotations/sroie_5....,receipt,CC BY 4.0 per HF card; verify in final reporting,...,463,605,355,1.0,0.546,35,35,"[""address"", ""company"", ""date"", ""total""]",downloaded_or_loaded_from_hf_cache,2026-04-29T09:47:44
6,sroie,data-sroie,jsdnrs/ICDAR2019-SROIE,train,6,sroie_6,data/raw/sandbox_v2/sroie/images/sroie_6.jpg,data/raw/sandbox_v2/sroie/annotations/sroie_6....,receipt,CC BY 4.0 per HF card; verify in final reporting,...,457,1170,902,1.0,0.456,93,93,"[""address"", ""company"", ""date"", ""total""]",downloaded_or_loaded_from_hf_cache,2026-04-29T09:47:44
7,sroie,data-sroie,jsdnrs/ICDAR2019-SROIE,train,7,sroie_7,data/raw/sandbox_v2/sroie/images/sroie_7.jpg,data/raw/sandbox_v2/sroie/annotations/sroie_7....,receipt,CC BY 4.0 per HF card; verify in final reporting,...,463,797,362,1.0,0.489,28,28,"[""address"", ""company"", ""date"", ""total""]",downloaded_or_loaded_from_hf_cache,2026-04-29T09:47:44
8,sroie,data-sroie,jsdnrs/ICDAR2019-SROIE,train,8,sroie_8,data/raw/sandbox_v2/sroie/images/sroie_8.jpg,data/raw/sandbox_v2/sroie/annotations/sroie_8....,receipt,CC BY 4.0 per HF card; verify in final reporting,...,992,1403,880,1.0,0.494,76,76,"[""address"", ""company"", ""date"", ""total""]",downloaded_or_loaded_from_hf_cache,2026-04-29T09:47:44
9,sroie,data-sroie,jsdnrs/ICDAR2019-SROIE,train,9,sroie_9,data/raw/sandbox_v2/sroie/images/sroie_9.jpg,data/raw/sandbox_v2/sroie/annotations/sroie_9....,receipt,CC BY 4.0 per HF card; verify in final reporting,...,604,1716,717,1.0,0.619,43,43,"[""address"", ""company"", ""date"", ""total""]",downloaded_or_loaded_from_hf_cache,2026-04-29T09:47:44


## NYPL Handoff For Notebook 2

`SBV2_02_nypl_sampling.ipynb` will build the menu-page sample. Notebook 1 only checks that the existing local NYPL files are available.

In [6]:
nypl_dir = ROOT / "data" / "raw" / "nypl"
nypl_files = {
    "Menu.csv": nypl_dir / "Menu.csv",
    "MenuPage.csv": nypl_dir / "MenuPage.csv",
    "MenuItem.csv": nypl_dir / "MenuItem.csv",
    "Dish.csv": nypl_dir / "Dish.csv",
}
nypl_image_dir = nypl_dir / "images"

nypl_rows = []
for name, path in nypl_files.items():
    line_count = None
    if path.exists():
        with path.open("r", encoding="utf-8", errors="ignore") as handle:
            line_count = sum(1 for _ in handle)
    nypl_rows.append(
        {
            "asset": name,
            "path": str(path.relative_to(ROOT)),
            "exists": path.exists(),
            "rows_including_header": line_count,
        }
    )

nypl_rows.append(
    {
        "asset": "images",
        "path": str(nypl_image_dir.relative_to(ROOT)),
        "exists": nypl_image_dir.exists(),
        "rows_including_header": sum(1 for p in nypl_image_dir.glob("*.jpg")) if nypl_image_dir.exists() else 0,
    }
)

nypl_handoff_df = pd.DataFrame(nypl_rows)
display(nypl_handoff_df)

nypl_ready = bool(nypl_handoff_df["exists"].all() and nypl_handoff_df.loc[nypl_handoff_df["asset"] == "images", "rows_including_header"].iloc[0] > 0)
print(f"NYPL handoff ready for SBV2_02: {nypl_ready}")

,asset,path,exists,rows_including_header
0,Menu.csv,data/raw/nypl/Menu.csv,True,17565
1,MenuPage.csv,data/raw/nypl/MenuPage.csv,True,66938
2,MenuItem.csv,data/raw/nypl/MenuItem.csv,True,1335256
3,Dish.csv,data/raw/nypl/Dish.csv,True,431099
4,images,data/raw/nypl/images,True,30


NYPL handoff ready for SBV2_02: True


## Optional And Deferred Sources

CORD is useful for item/price hierarchy stress testing, but it is Indonesian receipt data and stays outside the English generation claim. Korean menus are deferred entirely.

In [7]:
optional_rows = []

if CONFIG["include_cord"]:
    cord_spec = {
        "source_id": "cord",
        "register_id": "data-cord",
        "dataset_name": "naver-clova-ix/cord-v2",
        "split": "train",
        "limit": CONFIG["cord_limit"],
        "document_type": "receipt_ood_structure_stress",
        "licence": "CC BY 4.0 per project repo; verify in final reporting",
        "ground_truth_available": True,
    }
    cord_rows, cord_status = acquire_hf_sample(cord_spec)
    manifest_rows.extend(cord_rows)
    optional_rows.append(cord_status)
else:
    optional_rows.append(
        {
            "source_id": "cord",
            "dataset_name": "naver-clova-ix/cord-v2",
            "split": "train",
            "requested_limit": CONFIG["cord_limit"],
            "rows_staged": 0,
            "status": "deferred_by_default",
            "message": "Set SBV2_INCLUDE_CORD=1 only when a structure stress-test sidecar is needed.",
            "local_dir": str((RAW_DIR / "cord").relative_to(ROOT)),
        }
    )

optional_rows.append(
    {
        "source_id": "korean_menus",
        "dataset_name": "HumynLabs/Korean_Menus_Image_Dataset",
        "split": "future",
        "requested_limit": 0,
        "rows_staged": 0,
        "status": "deferred_future_ood",
        "message": "Not part of first English benchmark.",
        "local_dir": str((RAW_DIR / "korean_menus").relative_to(ROOT)),
    }
)

optional_df = pd.DataFrame(optional_rows)
if manifest_rows:
    manifest_df = pd.DataFrame(manifest_rows)

display(optional_df)

,source_id,dataset_name,split,requested_limit,rows_staged,status,message,local_dir
0,cord,naver-clova-ix/cord-v2,train,50,0,deferred_by_default,Set SBV2_INCLUDE_CORD=1 only when a structure ...,data/raw/sandbox_v2/cord
1,korean_menus,HumynLabs/Korean_Menus_Image_Dataset,future,0,0,deferred_future_ood,Not part of first English benchmark.,data/raw/sandbox_v2/korean_menus


## Persist Notebook 1 Outputs

In [8]:
if manifest_df.empty:
    source_counts_df = pd.DataFrame(columns=["source_id", "rows"])
else:
    source_counts_df = (
        manifest_df.groupby(["source_id", "document_type", "acquisition_status"], dropna=False)
        .size()
        .reset_index(name="rows")
        .sort_values(["source_id", "document_type", "acquisition_status"])
    )

combined_acquisition_df = pd.concat([acquisition_df, optional_df], ignore_index=True)

manifest_csv = INTERIM_MANIFEST_DIR / "sbv2_01_acquisition_manifest.csv"
manifest_parquet = INTERIM_MANIFEST_DIR / "sbv2_01_acquisition_manifest.parquet"
source_counts_csv = AUDIT_DIR / "sbv2_01_source_counts.csv"
acquisition_csv = AUDIT_DIR / "sbv2_01_dataset_acquisition.csv"
nypl_csv = AUDIT_DIR / "sbv2_01_nypl_handoff.csv"
summary_json = AUDIT_DIR / "sbv2_01_summary.json"

manifest_df.to_csv(manifest_csv, index=False)
try:
    manifest_df.to_parquet(manifest_parquet, index=False)
    parquet_status = "written"
except Exception as exc:
    parquet_status = f"not_written: {type(exc).__name__}: {exc}"

source_counts_df.to_csv(source_counts_csv, index=False)
combined_acquisition_df.to_csv(acquisition_csv, index=False)
nypl_handoff_df.to_csv(nypl_csv, index=False)

core_ready = bool(
    not manifest_df.empty
    and {"sroie", "funsd"}.issubset(set(manifest_df["source_id"].unique()))
    and nypl_ready
)

summary = {
    "created_at": RUN_TIMESTAMP,
    "project_root": str(ROOT),
    "config": CONFIG,
    "manifest_rows": int(len(manifest_df)),
    "source_counts": source_counts_df.to_dict(orient="records"),
    "nypl_ready_for_sbv2_02": nypl_ready,
    "core_ready_for_review": core_ready,
    "parquet_status": parquet_status,
    "outputs": {
        "manifest_csv": str(manifest_csv.relative_to(ROOT)),
        "manifest_parquet": str(manifest_parquet.relative_to(ROOT)),
        "source_counts_csv": str(source_counts_csv.relative_to(ROOT)),
        "acquisition_csv": str(acquisition_csv.relative_to(ROOT)),
        "nypl_handoff_csv": str(nypl_csv.relative_to(ROOT)),
    },
}
summary_json.write_text(json.dumps(jsonable(summary), indent=2), encoding="utf-8")

display(source_counts_df)
print("Wrote Notebook 1 outputs:")
for path in [manifest_csv, manifest_parquet, source_counts_csv, acquisition_csv, nypl_csv, summary_json]:
    print(f"- {path.relative_to(ROOT)}")
print(f"Parquet status: {parquet_status}")
print(f"Core ready for output review: {core_ready}")

,source_id,document_type,acquisition_status,rows
0,funsd,form,downloaded_or_loaded_from_hf_cache,149
1,sroie,receipt,downloaded_or_loaded_from_hf_cache,300


Wrote Notebook 1 outputs:
- data/interim/sandbox_v2/manifests/sbv2_01_acquisition_manifest.csv
- data/interim/sandbox_v2/manifests/sbv2_01_acquisition_manifest.parquet
- outputs/sandbox_v2/audit_tables/sbv2_01_source_counts.csv
- outputs/sandbox_v2/audit_tables/sbv2_01_dataset_acquisition.csv
- outputs/sandbox_v2/audit_tables/sbv2_01_nypl_handoff.csv
- outputs/sandbox_v2/audit_tables/sbv2_01_summary.json
Parquet status: written
Core ready for output review: True


## Review Gate Before SBV2 02

Discuss these outputs before creating Notebook 2:

- Are SROIE and FUNSD rows staged from HF cache/download, or did either fall back to smoke data?
- Are the staged row counts enough for the first pass, or should the limits be changed?
- Does the manifest schema contain enough fields for `SBV2_03_document_dataset_loading.ipynb`?
- Is the NYPL handoff healthy enough to sample menu pages in `SBV2_02_nypl_sampling.ipynb`?

Do not move to OCR or VLM notebooks until the manifest and NYPL sample decisions are settled.